# CSI-WaveNet — Multi-Resolution Fall Detection on CSI-Bench

**Self-contained Kaggle notebook.** Attach the [CSI-Bench dataset](https://www.kaggle.com/datasets/guozhenjennzhu/csi-bench) (Version 12+).

### Architecture: CSI-WaveNet
A novel multi-resolution wavelet-inspired convolutional network designed for WiFi CSI fall detection.

| Stage | Purpose | Signal Processing Interpretation |
|:------|:--------|:---------------------------------|
| Subcarrier Projection | 232 -> 64 | Learned bandpass filter bank over frequency-domain subcarriers |
| Multi-Resolution Branches | 3 parallel Conv1d (k=3,15,51) | Captures impact shock, body motion, posture transition |
| Temporal Compression | 2x ResBlock with stride-2 | Learned downsampling (500->250->125) |
| Attention Pooling | Softmax-weighted temporal sum | Learns WHEN the fall occurs in the 5s window |
| Classification Head | Linear + Dropout | Lightweight binary classifier |

**Targets:** >95% F1 (overall), >70% F1 (Hard), ~200K params, <5ms CPU latency

In [ ]:
# ============================================================
# Cell 1: Setup & Imports
# ============================================================
!pip install -q h5py onnx onnxruntime

import os, sys, json, random, time, math, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if not torch.cuda.is_available():
    raise RuntimeError('GPU required. Enable Kaggle GPU accelerator.')
torch.backends.cudnn.benchmark = True
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ============================================================
# Cell 2: Configuration — matches CSI-Bench protocol exactly
# ============================================================
WIN_LEN       = 500     # 5 seconds at ~100 Hz (CSI-Bench standard)
FEATURE_SIZE  = 232     # 116 amplitude + 116 phase subcarriers
BATCH_SIZE    = 64
EPOCHS        = 60
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
WARMUP_EPOCHS = 5
PATIENCE      = 12
DROPOUT       = 0.15
NUM_CLASSES   = 2       # Fall vs Non-Fall
MIXUP_ALPHA   = 0.3
OUT           = Path('/kaggle/working')

# --- CSI-Bench dataset path ---
FALL_DIR = Path('/kaggle/input/datasets/guozhenjennzhu/csi-bench/FallDetection')
if not FALL_DIR.exists():
    # fallback search
    for p in sorted(Path('/kaggle/input').rglob('FallDetection')):
        if p.is_dir():
            FALL_DIR = p; break

assert FALL_DIR.exists(), f'FallDetection dir not found. Tried: {FALL_DIR}'
SPLIT_DIR = FALL_DIR / 'splits'
print(f'FallDetection: {FALL_DIR}')
print(f'Splits exist:  {SPLIT_DIR.exists()}')

## Data Loading

Follows the official CSI-Bench data protocol:
- H5 files contain `CSI_amps` (116 subcarriers) and `CSI_phases` (116 subcarriers)
- Concatenated to form 232-dimensional feature vectors per time step
- Windowed/resampled to `win_len=500` frames (5 seconds at 100Hz)
- Uses official train/val/test splits from `splits/` directory when available

In [ ]:
# == Data loading (from working EdgeTCN notebook) ==
def read_csi(p):
    with h5py.File(str(p),'r') as f:
        ks=list(f.keys())
        if 'CSI_amps' in ks and 'CSI_phases' in ks:
            a=np.array(f['CSI_amps'],dtype=np.float32).squeeze()
            p_=np.array(f['CSI_phases'],dtype=np.float32).squeeze()
            if a.ndim!=2 or p_.ndim!=2: return None
            if a.shape[0]<a.shape[1]: a=a.T
            if p_.shape[0]<p_.shape[1]: p_=p_.T
            d=np.concatenate([a,p_],1)
        elif 'CSI_amps' in ks:
            d=np.array(f['CSI_amps'],dtype=np.float32).squeeze()
            if d.ndim!=2: return None
            if d.shape[0]<d.shape[1]: d=d.T
        else:
            d=np.array(f[ks[0]],dtype=np.float32).squeeze()
            if d.ndim!=2: return None
            if d.shape[0]<d.shape[1]: d=d.T
    d=np.nan_to_num(d,nan=0.,posinf=0.,neginf=0.)
    C=d.shape[1]
    if C>FEATURE_SIZE: d=d[:,:FEATURE_SIZE]
    elif C<FEATURE_SIZE: d=np.pad(d,((0,0),(0,FEATURE_SIZE-C)))
    L=d.shape[0]
    if L==WIN_LEN: return d
    elif L>WIN_LEN: s=(L-WIN_LEN)//2; return d[s:s+WIN_LEN]
    else:
        ot=np.linspace(0,1,L); nt=np.linspace(0,1,WIN_LEN)
        return np.stack([np.interp(nt,ot,d[:,k]) for k in range(FEATURE_SIZE)],1).astype(np.float32)

def act_label(p):
    for x in Path(str(p)).parts:
        xl=x.lower().replace('-','_')
        if xl.startswith('act_') and xl in ('act_fall','act_falling','act_fallen','act_fall_down'): return 1
    return 0

# Discover H5 files (same glob as working notebook)
files=sorted(FALL_DIR.glob('sub_Human/**/*.h5'))
if not files: files=sorted(FALL_DIR.rglob('*.h5'))
falls=[p for p in files if act_label(p)==1]; nonfalls=[p for p in files if act_label(p)==0]
print(f'H5: {len(files)} (Fall:{len(falls)} Non-Fall:{len(nonfalls)})')

# Try official splits by stem + multiple matching strategies
SPLIT_DIR=FALL_DIR/'splits'; splits={}
if SPLIT_DIR.exists():
    stem2p={p.stem:p for p in files}
    rel2p={}
    for p in files:
        try:
            r=str(p.relative_to(FALL_DIR)).replace('\\','/')
            rel2p[r]=p; rel2p[r.rsplit('.',1)[0]]=p
        except: pass
    for name in ['train','val','test','test_easy','test_medium','test_hard']:
        for sfx in ['_id.json','.json']:
            sp=SPLIT_DIR/f'{name}{sfx}'
            if not sp.exists(): continue
            ids=json.load(open(sp))
            if isinstance(ids,dict): ids=list(ids.values())[0] if ids else []
            if not isinstance(ids,list): continue
            matched=[]
            for sid in ids:
                sid=str(sid).replace('\\','/')
                fp=rel2p.get(sid) or stem2p.get(sid) or stem2p.get(Path(sid).stem)
                if fp is None:
                    for rp,rfp in rel2p.items():
                        if rp.endswith(sid) or sid.endswith(rp): fp=rfp; break
                if fp: matched.append(fp)
            if matched: splits[name]=matched; print(f'  {name:15s}: {len(matched)}')
            break
    if not splits or all(len(v)==0 for v in splits.values()):
        # Debug output
        for sfx in ['_id.json','.json']:
            sp=SPLIT_DIR/f'train{sfx}'
            if sp.exists():
                ids=json.load(open(sp))
                if isinstance(ids,dict): ids=list(ids.values())[0] if ids else []
                print(f'Split IDs sample: {ids[:3]}')
                break
        print(f'File stems sample: {[p.stem for p in files[:3]]}')
        try: print(f'File rels sample:  {[str(p.relative_to(FALL_DIR)).replace(chr(92),"/") for p in files[:3]]}')
        except: pass
        splits={}

# Fallback: random 80/20 split (same as working EdgeTCN notebook)
if not splits or 'train' not in splits or len(splits.get('train',[]))==0:
    print('Using random 80/20 split (same as EdgeTCN notebook)')
    rng=np.random.default_rng(SEED); f_=falls.copy(); n_=nonfalls.copy()
    rng.shuffle(f_); rng.shuffle(n_)
    cf=int(.8*len(f_)); cn=int(.8*len(n_))
    splits={'train':f_[:cf]+n_[:cn],'val':f_[cf:]+n_[cn:],'test':f_[cf:]+n_[cn:]}
for k in splits:
    if splits[k] and not isinstance(splits[k][0],tuple):
        splits[k]=[(p,act_label(p)) for p in splits[k]]
for k,v in splits.items(): print(f'  {k:15s}: {len(v)} (Fall:{sum(y for _,y in v)})')


In [ ]:
# == Build datasets ==
class CSIFallDataset(Dataset):
    def __init__(self, items, augment=False, stats=None):
        self.augment=augment; dl,ll=[],[]
        for i,(p,y) in enumerate(items):
            x=read_csi(p)
            if x is not None and x.shape==(WIN_LEN,FEATURE_SIZE): dl.append(x); ll.append(y)
            if (i+1)%500==0: print(f'  {i+1}/{len(items)}')
        self.data=np.stack(dl); self.labels=np.array(ll,dtype=np.int64)
        if stats is None:
            self.mean=self.data.mean(axis=(0,1),keepdims=True)
            self.std=self.data.std(axis=(0,1),keepdims=True)+1e-6
        else: self.mean,self.std=stats
    def get_stats(self): return (self.mean,self.std)
    def __len__(self): return len(self.data)
    def __getitem__(self,idx):
        x=self.data[idx].copy(); y=self.labels[idx]
        if self.augment:
            sh=np.random.randint(-50,51)
            if sh: x=np.roll(x,sh,0)
            x=x*np.random.uniform(.9,1.1,(1,FEATURE_SIZEURE_SIZE)).astype(np.float32)
            x=x+np.random.normal(0,.01,x.shape).astype(np.float32)
        x=(x-self.mean.squeeze())/self.std.squeeze()
        return torch.from_numpy(x).float(),torch.tensor(y,dtype=torch.long)

print('Loading train...'); train_ds=CSIFallDataset(splits['train'],augment=True)
ts=train_ds.get_stats()
print('Loading val...'); val_ds=CSIFallDataset(splits['val'],stats=ts)
test_datasets={}
for n in ['test','test_easy','test_medium','test_hard']:
    if n in splits and splits[n]: print(f'Loading {n}...'); test_datasets[n]=CSIFallDataset(splits[n],stats=ts)

tc=Counter(train_ds.labels.tolist()); print(f'Train:{len(train_ds)} Val:{len(val_ds)} Dist:{dict(tc)}')
cw={c:len(train_ds)/(NUM_CLASSES*n) for c,n in tc.items()}
sampler=WeightedRandomSampler([cw[y] for y in train_ds.labels.tolist()],len(train_ds),True)
train_loader=DataLoader(train_ds,BATCH_SIZE,sampler=sampler,num_workers=2,pin_memory=True,drop_last=True)
val_loader=DataLoader(val_ds,BATCH_SIZE*2,num_workers=2,pin_memory=True)
test_loaders={n:DataLoader(ds,BATCH_SIZE*2,num_workers=2,pin_memory=True) for n,ds in test_datasets.items()}


## CSI-WaveNet Architecture

```
Input: (B, 500, 232) — 5 seconds of amplitude + phase CSI
  |
  +-- SubcarrierProjection: Conv1d(232->64, k=1)
  |     "Which subcarriers carry information about human motion?"
  |
  +-- Multi-Resolution Extraction: 3 parallel Conv1d branches
  |     k=3  (30ms)  -> impact detection    --+
  |     k=15 (150ms) -> body motion           +-- concat -> (B, 96, 500)
  |     k=51 (510ms) -> posture transition  --+
  |
  +-- Temporal Compression: 2x ResBlock (stride-2)
  |     (B, 96, 500) -> (B, 128, 250) -> (B, 128, 125)
  |
  +-- Attention Pooling:
  |     Softmax attention + Max pool -> (B, 256)
  |
  +-- Classification Head: LN -> Linear(256,64) -> GELU -> Drop -> Linear(64,2)
```

In [ ]:
# ============================================================
# Cell 6: CSI-WaveNet Model Definition
# ============================================================

class ResBlock1d(nn.Module):
    """1D Residual Block with optional strided downsampling.
    
    Signal processing interpretation:
    - Two 1D convolutions extract local temporal patterns
    - Residual connection preserves input signal (prevents gradient vanishing)
    - Stride-2 provides 2x temporal compression via learned anti-aliased filter
    """
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm1d(out_ch)
        
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch)
            )
        else:
            self.skip = nn.Identity()
    
    def forward(self, x):
        identity = self.skip(x)
        out = F.gelu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return F.gelu(out + identity)


class CSIWaveNet(nn.Module):
    """Multi-Resolution Wavelet-Inspired Network for CSI Fall Detection.
    
    Architecture overview (each stage has a clear signal processing rationale):
    
    1. SUBCARRIER PROJECTION -- learned bandpass filter bank reducing
       232 frequency-domain subcarriers to 64 informative components.
    
    2. MULTI-RESOLUTION EXTRACTION -- three parallel temporal convolutions
       capture the full temporal profile of a fall:
       * k=3  (~30ms at 100Hz):  impact/shock wavelet
       * k=15 (~150ms at 100Hz): body motion pattern
       * k=51 (~510ms at 100Hz): posture transition (stand -> ground)
    
    3. TEMPORAL COMPRESSION -- residual blocks with stride-2 compress
       500 -> 250 -> 125 while building hierarchical features.
    
    4. ATTENTION POOLING -- learns WHEN the fall occurs within the window.
       Attention-weighted sum localizes the event; max pool captures peak.
    
    5. LIGHTWEIGHT HEAD -- single hidden layer prevents overfitting.
    
    ~200K parameters (50x smaller than ResNet-18).
    """
    
    def __init__(self, feature_size=232, num_classes=2, dropout=0.15):
        super().__init__()
        
        # Stage 1: Subcarrier Projection
        self.proj = nn.Sequential(
            nn.Conv1d(feature_size, 64, kernel_size=1, bias=False),
            nn.BatchNorm1d(64),
            nn.GELU()
        )
        
        # Stage 2: Multi-Resolution Feature Extraction
        self.branch_fast = nn.Sequential(
            nn.Conv1d(64, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(32), nn.GELU()
        )
        self.branch_mid = nn.Sequential(
            nn.Conv1d(64, 32, kernel_size=15, padding=7, bias=False),
            nn.BatchNorm1d(32), nn.GELU()
        )
        self.branch_slow = nn.Sequential(
            nn.Conv1d(64, 32, kernel_size=51, padding=25, bias=False),
            nn.BatchNorm1d(32), nn.GELU()
        )
        
        # Stage 3: Temporal Compression
        self.res1 = ResBlock1d(96, 128, stride=2)    # 500 -> 250
        self.res2 = ResBlock1d(128, 128, stride=2)   # 250 -> 125
        
        # Stage 4: Temporal Attention Pooling
        self.attn_score = nn.Conv1d(128, 1, kernel_size=1)
        
        # Stage 5: Classification Head
        self.head = nn.Sequential(
            nn.LayerNorm(256),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    
    def forward(self, x):
        # x: (B, 500, 232) -> (B, 232, 500) for Conv1d
        x = x.transpose(1, 2)
        
        # Stage 1: Project subcarriers
        x = self.proj(x)                               # (B, 64, 500)
        
        # Stage 2: Multi-resolution extraction
        fast = self.branch_fast(x)                     # (B, 32, 500)
        mid  = self.branch_mid(x)                      # (B, 32, 500)
        slow = self.branch_slow(x)                     # (B, 32, 500)
        x = torch.cat([fast, mid, slow], dim=1)        # (B, 96, 500)
        
        # Stage 3: Temporal compression
        x = self.res1(x)                               # (B, 128, 250)
        x = self.res2(x)                               # (B, 128, 125)
        
        # Stage 4: Attention pooling
        scores = self.attn_score(x)                    # (B, 1, 125)
        attn_w = F.softmax(scores, dim=-1)             # (B, 1, 125)
        attn_pool = (x * attn_w).sum(dim=-1)           # (B, 128)
        max_pool  = x.amax(dim=-1)                     # (B, 128)
        features  = torch.cat([attn_pool, max_pool], dim=1)  # (B, 256)
        
        # Stage 5: Classify
        return self.head(features)                     # (B, num_classes)


# --- Instantiate and inspect ---
model = CSIWaveNet(FEATURE_SIZE, NUM_CLASSES, DROPOUT).to(device)
n_params = sum(p.numel() for p in model.parameters())
n_train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'CSI-WaveNet created.')
print(f'  Total parameters:     {n_params:,}')
print(f'  Trainable parameters: {n_train_params:,}')
print(f'  vs ResNet-18:         ~11,000,000 ({n_params/11_000_000:.1%} the size)')

# Verify forward pass
with torch.no_grad():
    dummy = torch.randn(2, WIN_LEN, FEATURE_SIZE, device=device)
    out = model(dummy)
    print(f'  Input shape:  {dummy.shape}')
    print(f'  Output shape: {out.shape}')

In [ ]:
# ============================================================
# Cell 7: Training Utilities
# ============================================================

class FocalLoss(nn.Module):
    """Focal Loss for class-imbalanced classification (Lin et al. 2017).
    
    Down-weights easy examples, focuses on hard examples near decision boundary.
    Loss = -alpha_t * (1 - p_t)^gamma * log(p_t)
    When gamma=0, reduces to standard weighted cross-entropy.
    """
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float32))
        else:
            self.alpha = None
    
    def forward(self, inputs, targets):
        ce = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()


def mixup_data(x, y, alpha=0.3):
    """Mixup: blend pairs of inputs for smoother decision boundaries."""
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def get_cosine_schedule_with_warmup(optimizer, warmup_epochs, total_epochs):
    """Cosine annealing LR schedule with linear warmup."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return max(0.01, 0.5 * (1 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model on a DataLoader. Returns accuracy, per-class and weighted F1."""
    model.eval()
    all_preds, all_labels = [], []
    for x, y in loader:
        logits = model(x.to(device))
        all_preds.append(logits.argmax(dim=1).cpu())
        all_labels.append(y)
    preds  = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    acc = (preds == labels).mean()
    
    metrics = {'accuracy': float(acc)}
    for cls, name in [(0, 'nonfall'), (1, 'fall')]:
        tp = ((preds == cls) & (labels == cls)).sum()
        fp = ((preds == cls) & (labels != cls)).sum()
        fn = ((preds != cls) & (labels == cls)).sum()
        prec = tp / max(1, tp + fp)
        rec  = tp / max(1, tp + fn)
        f1   = 2 * prec * rec / max(1e-9, prec + rec)
        metrics[f'{name}_precision'] = float(prec)
        metrics[f'{name}_recall']    = float(rec)
        metrics[f'{name}_f1']        = float(f1)
    
    counts = Counter(labels.tolist())
    total = sum(counts.values())
    weighted_f1 = sum(
        metrics[f"{('nonfall','fall')[c]}_f1"] * counts.get(c, 0) / total
        for c in [0, 1]
    )
    metrics['weighted_f1'] = float(weighted_f1)
    return metrics


# --- Setup loss, optimizer, scheduler ---
class_freqs = torch.tensor(
    [train_counts.get(0, 1), train_counts.get(1, 1)], dtype=torch.float32
)
class_alpha = (1.0 / class_freqs)
class_alpha = class_alpha / class_alpha.sum() * NUM_CLASSES
print(f'Focal Loss class weights: {class_alpha.tolist()}')

criterion = FocalLoss(alpha=class_alpha.to(device), gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP_EPOCHS, EPOCHS)
print('Training utilities ready.')

In [ ]:
# ============================================================
# Cell 8: Training Loop
# ============================================================

best_val_f1 = -1.0
best_state  = None
best_metrics = None
stale = 0
history = []

print(f'Training CSI-WaveNet for up to {EPOCHS} epochs (patience={PATIENCE})')
print(f'Mixup enabled after warmup (epoch {WARMUP_EPOCHS+1}+), alpha={MIXUP_ALPHA}')
print('=' * 90)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    total_loss = 0.0
    n_samples = 0
    use_mixup = (epoch > WARMUP_EPOCHS)
    
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        
        if use_mixup:
            x_mix, y_a, y_b, lam = mixup_data(x, y, MIXUP_ALPHA)
            logits = model(x_mix)
            loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
        else:
            logits = model(x)
            loss = criterion(logits, y)
        
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item() * x.size(0)
        n_samples += x.size(0)
    
    scheduler.step()
    avg_loss = total_loss / max(1, n_samples)
    
    val_m = evaluate(model, val_loader)
    elapsed = time.time() - t0
    
    history.append({
        'epoch': epoch, 'loss': avg_loss,
        'val_acc': val_m['accuracy'], 'val_f1': val_m['weighted_f1'],
        'val_fall_f1': val_m['fall_f1'],
        'lr': optimizer.param_groups[0]['lr']
    })
    
    marker = ''
    if val_m['weighted_f1'] > best_val_f1:
        best_val_f1 = val_m['weighted_f1']
        best_metrics = val_m.copy()
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
        marker = ' * BEST'
    else:
        stale += 1
    
    print(
        f'Epoch {epoch:3d}/{EPOCHS} | '
        f'loss={avg_loss:.4f} | '
        f'val_acc={val_m["accuracy"]:.4f} | '
        f'val_F1={val_m["weighted_f1"]:.4f} | '
        f'fall_F1={val_m["fall_f1"]:.4f} | '
        f'lr={optimizer.param_groups[0]["lr"]:.2e} | '
        f'{elapsed:.1f}s{marker}'
    )
    
    if stale >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

model.load_state_dict(best_state)
print(f'\n{"=" * 90}')
print(f'Best validation weighted F1: {best_val_f1:.4f}')
print(json.dumps(best_metrics, indent=2))

In [ ]:
# ============================================================
# Cell 9: Evaluation on All Test Splits
# ============================================================

print('\n' + '=' * 90)
print('TEST SET EVALUATION (best model checkpoint)')
print('=' * 90)

all_results = {}

# CSI-Bench baselines (from paper + RESULTS.md, overall test)
baselines = {
    'MLP':           {'acc': 92.16, 'f1': 92.17},
    'LSTM':          {'acc': 94.93, 'f1': 94.92},
    'ResNet-18':     {'acc': 94.88, 'f1': 94.89},
    'Transformer':   {'acc': 94.28, 'f1': 94.26},
    'PatchTST':      {'acc': 94.03, 'f1': 94.03},
    'TimesFormer1D': {'acc': 93.86, 'f1': 93.87},
    'ViT':           {'acc': 93.58, 'f1': 93.59},
}

for split_name, loader in test_loaders.items():
    m = evaluate(model, loader)
    all_results[split_name] = m
    print(f'\n--- {split_name.upper()} ---')
    print(f'  Accuracy:        {m["accuracy"]*100:.2f}%')
    print(f'  Weighted F1:     {m["weighted_f1"]*100:.2f}%')
    print(f'  Fall Precision:  {m["fall_precision"]*100:.2f}%')
    print(f'  Fall Recall:     {m["fall_recall"]*100:.2f}%')
    print(f'  Fall F1:         {m["fall_f1"]*100:.2f}%')

if 'test' in all_results:
    our_f1 = all_results['test']['weighted_f1'] * 100
    our_acc = all_results['test']['accuracy'] * 100
    print(f'\n{"=" * 90}')
    print(f'COMPARISON vs CSI-Bench BASELINES (overall test weighted F1)')
    print(f'{"=" * 90}')
    print(f'{"Model":<18s} {"Acc (%)":>10s} {"F1 (%)":>10s}')
    print('-' * 40)
    for name, b in baselines.items():
        print(f'{name:<18s} {b["acc"]:>10.2f} {b["f1"]:>10.2f}')
    print('-' * 40)
    print(f'{"CSI-WaveNet (ours)":<18s} {our_acc:>10.2f} {our_f1:>10.2f}')
    print(f'\nCSI-WaveNet params: {n_params:,} vs ResNet-18: ~11,000,000')

In [ ]:
# ============================================================
# Cell 10: Save Model + ONNX Export + Latency Benchmark
# ============================================================

# Save PyTorch checkpoint
ckpt_path = OUT / 'csi_wavenet_best.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'architecture': 'CSIWaveNet',
    'feature_size': FEATURE_SIZE,
    'win_len': WIN_LEN,
    'num_classes': NUM_CLASSES,
    'num_params': n_params,
    'best_val_metrics': best_metrics,
    'test_results': all_results,
    'training_history': history,
    'config': {
        'seed': SEED, 'epochs': EPOCHS, 'lr': LR,
        'weight_decay': WEIGHT_DECAY, 'dropout': DROPOUT,
        'batch_size': BATCH_SIZE, 'mixup_alpha': MIXUP_ALPHA
    },
    'normalization': {
        'mean': train_ds.mean.tolist(),
        'std': train_ds.std.tolist()
    }
}, ckpt_path)
print(f'Checkpoint saved: {ckpt_path} ({ckpt_path.stat().st_size/1024:.1f} KB)')

# ONNX Export
cpu_model = model.to('cpu').eval()
onnx_path = OUT / 'csi_wavenet_best.onnx'

try:
    dummy_input = torch.randn(1, WIN_LEN, FEATURE_SIZE)
    torch.onnx.export(
        cpu_model, dummy_input, str(onnx_path),
        input_names=['csi_input'],
        output_names=['logits'],
        dynamic_axes={'csi_input': {0: 'batch'}, 'logits': {0: 'batch'}},
        opset_version=17
    )
    print(f'ONNX exported: {onnx_path} ({onnx_path.stat().st_size/1024:.1f} KB)')
    
    # Validate ONNX matches PyTorch
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    sample = np.random.randn(1, WIN_LEN, FEATURE_SIZE).astype(np.float32)
    pt_out = cpu_model(torch.from_numpy(sample)).detach().numpy()
    onnx_out = sess.run(['logits'], {'csi_input': sample})[0]
    max_diff = float(np.max(np.abs(pt_out - onnx_out)))
    print(f'ONNX validation: max |PyTorch - ONNX| = {max_diff:.2e}')
    
    # Latency benchmark
    latencies = []
    for _ in range(100):
        t0 = time.perf_counter()
        sess.run(['logits'], {'csi_input': sample})
        latencies.append((time.perf_counter() - t0) * 1000)
    print(f'\nCPU Latency (100 runs):')
    print(f'  Median: {np.median(latencies):.2f} ms')
    print(f'  P95:    {np.percentile(latencies, 95):.2f} ms')
    print(f'  P99:    {np.percentile(latencies, 99):.2f} ms')

except Exception as e:
    print(f'ONNX export/validation failed: {type(e).__name__}: {e}')

model.to(device)
print('\nDone!')

In [ ]:
# ============================================================
# Cell 11: Results Summary JSON
# ============================================================

summary = {
    'architecture': 'CSIWaveNet',
    'total_params': n_params,
    'dataset': 'CSI-Bench/FallDetection',
    'input_shape': [WIN_LEN, FEATURE_SIZE],
    'best_val_weighted_f1': best_metrics['weighted_f1'],
    'test_results': {
        k: {mk: round(mv, 4) for mk, mv in v.items()}
        for k, v in all_results.items()
    },
    'csi_bench_baselines': baselines,
    'config': {
        'seed': SEED, 'epochs': EPOCHS, 'lr': LR,
        'weight_decay': WEIGHT_DECAY, 'dropout': DROPOUT,
        'batch_size': BATCH_SIZE, 'mixup_alpha': MIXUP_ALPHA,
        'warmup_epochs': WARMUP_EPOCHS, 'patience': PATIENCE,
        'focal_gamma': 2.0
    },
    'hardware_compatibility': {
        'esp32_feature_size': 232,
        'esp32_format': 'CSI_TENSOR,timestamp,frame_id,amp_0,...,amp_115,phase_0,...,phase_115',
        'sampling_rate_hz': 100,
        'window_frames': WIN_LEN
    }
}

summary_path = OUT / 'csi_wavenet_results.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print(f'\nSaved to: {summary_path}')

## ESP32 Hardware Compatibility

This model is designed to work directly with the Sannidhi ESP32 firmware:

| Parameter | CSI-Bench Training | ESP32 Hardware |
|:----------|:-------------------|:---------------|
| Feature size | 232 (116 amp + 116 phase) | 232 (matches `scan.c` output) |
| Sampling rate | ~100 Hz | Set `ping.py` HZ=100 |
| Window length | 500 frames (5 seconds) | Collect 500 `CSI_TENSOR` rows per window |
| Preprocessing | Z-score normalization | Apply `mean`/`std` from checkpoint |

### Deployment Steps
1. Flash updated ESP32 firmware (same `scan.c`, updated `ping.py` to 100Hz)
2. Record 500 frames using `recorder.py`
3. Parse CSV: extract 232 features per row, stack into (500, 232) array
4. Normalize using saved `mean`/`std` from checkpoint
5. Run inference through ONNX model

In [ ]:
# ============================================================
# Visualization
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, roc_curve, auc

def plot_results(eval_model, loader, is_bce, threshold=0.5):
    eval_model.eval()
    all_p, all_y = [], []
    with torch.no_grad():
        for x, y in loader:
            out = eval_model(x.to(device))
            if is_bce:
                p = torch.sigmoid(out[:, 0]).cpu().numpy()
                y_true = y[:, 0].cpu().numpy() if y.ndim > 1 else y.cpu().numpy()
            else:
                p = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
                y_true = y.cpu().numpy()
            all_p.extend(p)
            all_y.extend(y_true)
            
    all_p, all_y = np.array(all_p), np.array(all_y)
    preds = (all_p >= threshold).astype(int)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    
    # Confusion Matrix
    cm = confusion_matrix(all_y, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1, 
                xticklabels=['Non-Fall', 'Fall'], yticklabels=['Non-Fall', 'Fall'])
    ax1.set_title('Test Split Confusion Matrix')
    ax1.set_ylabel('True Label')
    ax1.set_xlabel('Predicted Label')
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(all_y, all_p)
    ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {auc(fpr, tpr):.3f}')
    ax2.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    ax2.set_xlabel('False Positive Rate'); ax2.set_ylabel('True Positive Rate')
    ax2.set_title('ROC Curve')
    ax2.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

# Auto-detect parameters based on notebook
target_model = chosen if 'chosen' in locals() else model
bce_mode = 'chosen' in locals()  # FallNet uses 'chosen' and BCE
thr = chosen_m['threshold'] if 'chosen_m' in locals() else 0.5

if 'te_loaders' in locals() and 'test' in te_loaders:
    eval_loader = te_loaders['test']
elif 'test_loaders' in locals() and 'test' in test_loaders:
    eval_loader = test_loaders['test']
else:
    eval_loader = val_loader

print("Generating visual evaluation...")
plot_results(target_model, eval_loader, bce_mode, thr)
